# Prepare K-to-H (POI to Home) Data (Refactored)

**Changes from 5-sp-accessibility-kh-prep.ipynb:**
- Simplified: no pre-grouping by time bins
- Identifies reachable POIs from WK results
- Prepares single origins file (all reachable POIs) for script 6b

In [1]:
%load_ext autoreload
%autoreload 2
%cd D:\netmob25

D:\netmob25


In [2]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

## Configuration

In [3]:
# Directories
DATA_DIR = Path("dbs/sp_accessibility_v2")
INPUT_DIR = DATA_DIR
OUTPUT_DIR = DATA_DIR / "data"

# Departure hours (should match 4b)
DEPARTURE_HOURS = [16, 17, 18]

# Modes
MODES = ['pt', 'car']

print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

Input directory: dbs\sp_accessibility_v2
Output directory: dbs\sp_accessibility_v2\data


## 1. Load time budget data

In [4]:
df_budget = pd.read_csv(OUTPUT_DIR / "time_budget.csv")
print(f"Time budget data: {len(df_budget)} individuals")
df_budget.head()

Time budget data: 2457 individuals


,ID,time_budget,time_hw,tt_wkh,is_car
0,10_2978,97.0,25.0,40.0,0
1,10_2980,52.0,21.0,48.0,0
2,10_2981,48.0,43.5,3.0,0
3,10_2982,201.0,106.0,-122.0,1
4,10_2984,36.0,20.0,50.0,1


## 2. Process WK travel time results

For each mode and departure hour, identify:
- Which individuals can reach at least one POI
- Which POIs are reachable by at least one individual

In [5]:
MAX_BUDGET = 120  # Use max budget to capture all potentially reachable POIs

def process_wk_results(mode, hour):
    """
    Load WK travel time results and identify reachable POIs.
    
    Uses MAX_BUDGET (120 min) for filtering to ensure all POIs reachable
    under any sensitivity scenario are included. Actual budget filtering
    happens in 7b post-processing.
    
    Returns:
        df: Travel times with remaining time budget
        reachable_pois: Set of POI IDs reachable by at least one individual
        reachable_individuals: Set of individual IDs who can reach at least one POI
    """
    tt_file = INPUT_DIR / f"tt_wk_{mode}_{hour:02d}.csv"
    
    if not tt_file.exists():
        print(f"Warning: {tt_file} not found")
        return None, set(), set()
    
    print(f"Loading {tt_file}...")
    df = pd.read_csv(tt_file)
    df.rename(columns={
        'from_id': 'ID',
        'to_id': 'poi_id',
        'travel_time_p50': 'time_wk'
    }, inplace=True)
    
    print(f"  Loaded {len(df)} O-D pairs")
    print(f"  Individuals: {df['ID'].nunique()}")
    print(f"  POIs: {df['poi_id'].nunique()}")
    
    # Merge with time budget to get time_hw
    df = df.merge(df_budget[['ID', 'time_hw']], on='ID', how='left')
    
    # Compute remaining time using MAX budget (120 min)
    # This ensures all POIs reachable under any sensitivity scenario are included
    df['time_remaining_max'] = (MAX_BUDGET - df['time_hw'] * 2) - df['time_wk']
    
    # Filter to reachable pairs (positive remaining time under max budget)
    df_reachable = df[df['time_remaining_max'] > 0]
    
    reachable_pois = set(df_reachable['poi_id'].unique())
    reachable_individuals = set(df_reachable['ID'].unique())
    
    print(f"  Reachable POIs (budget={MAX_BUDGET}): {len(reachable_pois)}")
    print(f"  Individuals with reachable POIs: {len(reachable_individuals)}")
    
    return df, reachable_pois, reachable_individuals

In [6]:
# Collect all reachable POIs and individuals across modes and hours
all_reachable_pois = {}
all_reachable_individuals = {}

for mode in MODES:
    for hour in DEPARTURE_HOURS:
        key = f"{mode}_{hour:02d}"
        print(f"\n=== Processing {key} ===")
        
        _, pois, individuals = process_wk_results(mode, hour)
        all_reachable_pois[key] = pois
        all_reachable_individuals[key] = individuals


=== Processing pt_16 ===
Loading dbs\sp_accessibility_v2\tt_wk_pt_16.csv...
  Loaded 27663158 O-D pairs
  Individuals: 1156
  POIs: 42747
  Reachable POIs (budget=120): 42406
  Individuals with reachable POIs: 1143

=== Processing pt_17 ===
Loading dbs\sp_accessibility_v2\tt_wk_pt_17.csv...
  Loaded 28146850 O-D pairs
  Individuals: 1156
  POIs: 42828
  Reachable POIs (budget=120): 42564
  Individuals with reachable POIs: 1143

=== Processing pt_18 ===
Loading dbs\sp_accessibility_v2\tt_wk_pt_18.csv...
  Loaded 28190148 O-D pairs
  Individuals: 1156
  POIs: 42844
  Reachable POIs (budget=120): 42561
  Individuals with reachable POIs: 1143

=== Processing car_16 ===
Loading dbs\sp_accessibility_v2\tt_wk_car_16.csv...
  Loaded 25178569 O-D pairs
  Individuals: 662
  POIs: 43575
  Reachable POIs (budget=120): 43575
  Individuals with reachable POIs: 660

=== Processing car_17 ===
Loading dbs\sp_accessibility_v2\tt_wk_car_17.csv...
  Loaded 25178569 O-D pairs
  Individuals: 662
  POIs: 43

## 3. Prepare origins for KH leg (reachable POIs)

In [7]:
# Load POI coordinates
df_pois = pd.read_csv(OUTPUT_DIR / "destinations_leisure.csv")
print(f"Total LEISURE POIs: {len(df_pois)}")

Total LEISURE POIs: 43576


In [8]:
# For each mode-hour combination, save reachable POIs as origins
for mode in MODES:
    for hour in DEPARTURE_HOURS:
        key = f"{mode}_{hour:02d}"
        reachable = all_reachable_pois[key]
        
        if len(reachable) == 0:
            print(f"No reachable POIs for {key}, skipping")
            continue
        
        # Filter POIs to reachable ones
        df_origins = df_pois[df_pois['id'].isin(reachable)].copy()
        
        # Save
        out_file = OUTPUT_DIR / f"origins_kh_{key}.csv"
        df_origins[['id', 'lon', 'lat']].to_csv(out_file, index=False)
        print(f"Saved {len(df_origins)} POIs to {out_file}")

Saved 42406 POIs to dbs\sp_accessibility_v2\data\origins_kh_pt_16.csv
Saved 42564 POIs to dbs\sp_accessibility_v2\data\origins_kh_pt_17.csv
Saved 42561 POIs to dbs\sp_accessibility_v2\data\origins_kh_pt_18.csv
Saved 43575 POIs to dbs\sp_accessibility_v2\data\origins_kh_car_16.csv
Saved 43575 POIs to dbs\sp_accessibility_v2\data\origins_kh_car_17.csv
Saved 43575 POIs to dbs\sp_accessibility_v2\data\origins_kh_car_18.csv


## 4. Prepare destinations for KH leg (home locations)

In [9]:
# Load home coordinates
df_homes = pd.read_csv(OUTPUT_DIR / "destinations_home.csv")
print(f"Total home locations: {len(df_homes)}")

Total home locations: 2453


In [10]:
# For each mode-hour combination, save relevant home locations as destinations
for mode in MODES:
    for hour in DEPARTURE_HOURS:
        key = f"{mode}_{hour:02d}"
        reachable_ind = all_reachable_individuals[key]
        
        if len(reachable_ind) == 0:
            print(f"No reachable individuals for {key}, skipping")
            continue
        
        # Filter homes to those of individuals with reachable POIs
        df_dests = df_homes[df_homes['id'].isin(reachable_ind)].copy()
        
        # Save
        out_file = OUTPUT_DIR / f"destinations_kh_{key}.csv"
        df_dests[['id', 'lon', 'lat']].to_csv(out_file, index=False)
        print(f"Saved {len(df_dests)} homes to {out_file}")

Saved 1142 homes to dbs\sp_accessibility_v2\data\destinations_kh_pt_16.csv
Saved 1142 homes to dbs\sp_accessibility_v2\data\destinations_kh_pt_17.csv
Saved 1142 homes to dbs\sp_accessibility_v2\data\destinations_kh_pt_18.csv
Saved 660 homes to dbs\sp_accessibility_v2\data\destinations_kh_car_16.csv
Saved 660 homes to dbs\sp_accessibility_v2\data\destinations_kh_car_17.csv
Saved 660 homes to dbs\sp_accessibility_v2\data\destinations_kh_car_18.csv


## Summary

In [11]:
print("=" * 50)
print("KH preparation complete!")
print("=" * 50)
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"\nFiles created per mode-hour combination:")
print(f"  - origins_kh_{{mode}}_{{hour}}.csv: Reachable POIs")
print(f"  - destinations_kh_{{mode}}_{{hour}}.csv: Home locations")
print(f"\nNext step: Run 6b-sp-accessibility-kh.R")

KH preparation complete!

Output directory: dbs\sp_accessibility_v2\data

Files created per mode-hour combination:
  - origins_kh_{mode}_{hour}.csv: Reachable POIs
  - destinations_kh_{mode}_{hour}.csv: Home locations

Next step: Run 6b-sp-accessibility-kh.R
